# Fluor Unified: Fluorescent Molecule Property Prediction

In [ ]:
import subprocess, sys, os
import torch

# Detect CUDA version for DGL wheel selection
cuda_version = None
if torch.cuda.is_available():
    cuda_version = torch.version.cuda
    print(f"CUDA {cuda_version} detected")
else:
    print("No CUDA detected, using CPU")

# Install DGL from the correct wheel URL
# DGL 2.4 supports up to PyTorch 2.4 (ref: https://github.com/dmlc/dgl/issues/7822)
torch_ver = ".".join(torch.__version__.split(".")[:2])
if cuda_version:
    dgl_url = f"https://data.dgl.ai/wheels/torch-{torch_ver}/cu{cuda_version.replace('.','')}/repo.html"
else:
    dgl_url = f"https://data.dgl.ai/wheels/torch-{torch_ver}/repo.html"

subprocess.run([sys.executable, "-m", "pip", "install", "dgl", "-f", dgl_url], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "rdkit", "dgllife", "tqdm"], check=False)

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import torch._dynamo
torch._dynamo.config.suppress_errors = True
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO_URL = "https://github.com/your-org/fluor-tools.git"  # placeholder
REPO_DIR = "/content/fluor-tools"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

# Add to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Set up data directories
DRIVE_ROOT = "/content/drive/MyDrive/fluor-tools"
DATASETS_DIR = f"{DRIVE_ROOT}/datasets"
ACTIVE_RUN_DIR = f"{DRIVE_ROOT}/training-runs/active"
COMPLETED_RUNS_DIR = f"{DRIVE_ROOT}/training-runs/completed"
for d in [DATASETS_DIR, ACTIVE_RUN_DIR, COMPLETED_RUNS_DIR]:
    os.makedirs(d, exist_ok=True)
print("Setup complete")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

mode_selector = widgets.RadioButtons(
    options=["Create Training Data", "Predict Properties", "Train Models"],
    description="Mode:",
    layout=widgets.Layout(width="400px"),
)
display(mode_selector)

In [ ]:
# Run this cell when mode == "Create Training Data"
from colab_training.fluor_modules import data_pipeline

csv_path_input = widgets.Text(
    value="",
    placeholder="/content/drive/MyDrive/fluor-tools/datasets/my_data.csv",
    description="CSV path:",
    layout=widgets.Layout(width="600px"),
)
display(csv_path_input)

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = csv_path_input.value.strip()
if not csv_path:
    print("Please enter a CSV path above")
else:
    solvent_mapping = data_pipeline.load_solvent_mapping(
        f"{REPO_DIR}/Fluor-RLAT/data/00_solvent_mapping.csv"
    )
    patterns = data_pipeline.load_substructure_patterns(
        f"{REPO_DIR}/Fluor-RLAT/data/00_mmp_substructure.csv"
    )
    main_df, smiles_fp_df, sol_fp_df, skipped = data_pipeline.process_input_csv(
        csv_path, solvent_mapping, patterns
    )
    print(f"Processed {len(main_df)} valid rows, {len(skipped)} skipped")
    if skipped:
        print("Skipped rows:")
        for name, reason in skipped:
            print(f"  {name}: {reason}")
    display(main_df.head())

In [ ]:
# Merge with existing training data and save to Google Drive
split_data = data_pipeline.split_by_property(main_df, smiles_fp_df, sol_fp_df)
for prop, (prop_main, prop_smiles, prop_sol) in split_data.items():
    existing_main_path = f"{DATASETS_DIR}/train_{prop}.csv"
    if os.path.exists(existing_main_path):
        existing_main = pd.read_csv(existing_main_path)
        existing_smiles = pd.read_csv(f"{DATASETS_DIR}/train_smiles_{prop}.csv")
        existing_sol = pd.read_csv(f"{DATASETS_DIR}/train_sol_{prop}.csv")
        merged_main, merged_smiles, merged_sol = data_pipeline.merge_training_data(
            prop_main, prop_smiles, prop_sol,
            existing_main, existing_smiles, existing_sol,
        )
        print(f"{prop}: merged {len(prop_main)} new + {len(existing_main)} existing = {len(merged_main)} rows")
    else:
        merged_main, merged_smiles, merged_sol = prop_main, prop_smiles, prop_sol
        print(f"{prop}: {len(merged_main)} new rows (no existing data)")
    merged_main.to_csv(f"{DATASETS_DIR}/train_{prop}.csv", index=False)
    merged_smiles.to_csv(f"{DATASETS_DIR}/train_smiles_{prop}.csv", index=False)
    merged_sol.to_csv(f"{DATASETS_DIR}/train_sol_{prop}.csv", index=False)
print("Saved to Google Drive")

In [ ]:
model_source = widgets.RadioButtons(
    options=["pretrained", "custom"],
    description="Model source:",
)
display(model_source)

In [ ]:
smiles_input = widgets.Textarea(
    value="",
    placeholder="Enter SMILES (one per line) or leave empty to use CSV",
    description="SMILES:",
    layout=widgets.Layout(width="600px", height="100px"),
)
solvent_input = widgets.Text(
    value="EtOH",
    description="Solvent:",
    layout=widgets.Layout(width="300px"),
)
display(smiles_input, solvent_input)

In [ ]:
from colab_training.fluor_modules import prediction_engine

# Resolve model directory
if model_source.value == "pretrained":
    model_dir = f"{REPO_DIR}/Fluor-RLAT"
else:
    model_dir = prediction_engine.find_latest_completed_run(COMPLETED_RUNS_DIR)
    if model_dir is None:
        print("No custom models found. Please train models first or use pretrained.")
        model_dir = None

if model_dir:
    data_dir = f"{REPO_DIR}/Fluor-RLAT/data"
    from fluor_modules.data_pipeline import map_solvent_name_to_smiles
    sol_smiles = map_solvent_name_to_smiles(solvent_input.value) or solvent_input.value

    smiles_list = [s.strip() for s in smiles_input.value.strip().split("\n") if s.strip()]
    if not smiles_list:
        print("Please enter at least one SMILES string")
    else:
        results = []
        for smi in smiles_list:
            preds = prediction_engine.predict_all_properties(smi, sol_smiles, model_dir, data_dir, device)
            preds["smiles"] = smi
            results.append(preds)
        results_df = pd.DataFrame(results)
        display(results_df)

In [ ]:
from colab_training.fluor_modules import training_engine
import uuid

prop_checkboxes = {
    p: widgets.Checkbox(value=True, description=p)
    for p in ["abs", "em", "plqy", "k"]
}
data_source_selector = widgets.RadioButtons(
    options=["original repo data", "merged Google Drive data"],
    description="Data source:",
)
display(*prop_checkboxes.values(), data_source_selector)

In [ ]:
selected_targets = [p for p, cb in prop_checkboxes.items() if cb.value]
if not selected_targets:
    print("Please select at least one property to train")
else:
    if data_source_selector.value == "merged Google Drive data":
        data_dir = DATASETS_DIR
    else:
        data_dir = f"{REPO_DIR}/Fluor-RLAT/data"

    from fluor_modules.config import MODEL_REGISTRY, LEARNING_RATE, BATCH_SIZE
    model_configs = {t: MODEL_REGISTRY[t] for t in selected_targets}
    run_config = training_engine.TrainingConfig(
        targets=selected_targets,
        epochs=200,
        patience=30,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        model_configs=model_configs,
        data_source=data_dir,
        lr_scheduler_factor=0.5,
        lr_scheduler_patience=10,
        lr_scheduler_min=1e-6,
        run_id=str(uuid.uuid4()),
    )

    session_status = training_engine.check_existing_session(ACTIVE_RUN_DIR, run_config)
    if session_status == "mismatch":
        print("WARNING: Existing session has different config. Starting fresh.")
    elif session_status == "match":
        print("Resuming existing session...")

    # Save config
    import json
    os.makedirs(ACTIVE_RUN_DIR, exist_ok=True)
    with open(f"{ACTIVE_RUN_DIR}/training_config.json", "w") as f:
        f.write(run_config.to_json())

    all_metrics = {}
    for target in selected_targets:
        print(f"\nTraining {target}...")
        metrics = training_engine.train_model(
            target=target,
            data_dir=data_dir,
            active_dir=ACTIVE_RUN_DIR,
            config=MODEL_REGISTRY[target],
            epochs=run_config.epochs,
            patience=run_config.patience,
            learning_rate=run_config.learning_rate,
            device=device,
        )
        all_metrics[target] = metrics
        print(f"{target}: MAE={metrics['mae']:.4f}, RMSE={metrics['rmse']:.4f}, R2={metrics['r2']:.4f}")

    archive_path = training_engine.archive_completed_run(ACTIVE_RUN_DIR, COMPLETED_RUNS_DIR, selected_targets)
    print(f"\nTraining complete. Models archived to: {archive_path}")
    display(pd.DataFrame(all_metrics).T[["mae", "rmse", "r2"]])